In [1]:
# NB : This reload your library after each edit, 
# so you don't have to restart the kernel
%load_ext autoreload
%autoreload 2



In [21]:
import mlflow
import pandas as pd
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("diamonds")
from diamonds.data import load_data, clean_data, create_X_y, split_data
from diamonds.model import create_model, create_preproc, train_model, evaluate_model, predict, preprocess_data, run_model_mkflow
from diamonds.registry import load_model_mkflow
from diamonds.predict import predict_values
import random

In [3]:
df = load_data()
df

2026-03-10 15:55:40.950 | INFO     | diamonds.data:load_data:24 - Loading the data ...
2026-03-10 15:55:40.951 | INFO     | diamonds.data:load_data:28 - Data not found in cache, loading from seaborn ...


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
...,...,...,...,...,...,...,...,...,...,...
53935,0.72,Ideal,D,SI1,60.8,57.0,2757,5.75,5.76,3.50
53936,0.72,Good,D,SI1,63.1,55.0,2757,5.69,5.75,3.61
53937,0.70,Very Good,D,SI1,62.8,60.0,2757,5.66,5.68,3.56
53938,0.86,Premium,H,SI2,61.0,58.0,2757,6.15,6.12,3.74


In [4]:
df_clean = clean_data(df)
len(df_clean)

2026-03-10 15:55:45.608 | INFO     | diamonds.data:clean_data:61 - Removed 20 rows with null values


53920

In [5]:
df = load_data()
df_clean = clean_data(df)
X, y = create_X_y(df_clean)
X_train, X_test, y_train, y_test = split_data(X, y)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

2026-03-10 15:55:50.115 | INFO     | diamonds.data:load_data:24 - Loading the data ...
2026-03-10 15:55:50.116 | INFO     | diamonds.data:load_data:35 - Data found in cache, loading from cache ...


2026-03-10 15:55:50.347 | INFO     | diamonds.data:clean_data:61 - Removed 20 rows with null values


(43136, 9) (10784, 9) (43136,) (10784,)


In [7]:
preproc = create_preproc(X_train)
X_train_scaled = preprocess_data(preproc, X_train)
X_test_scaled = preprocess_data(preproc, X_test)
model_ = create_model("random_forest")
model_ = train_model(model_, X_train_scaled, y_train)
y_pred = predict(model_, X_test_scaled)
evaluate_model(y_test, y_pred)


2026-03-10 15:56:55.708 | INFO     | diamonds.model:create_preproc:68 - Preprocessing pipeline created and fitted on the training data
2026-03-10 15:56:55.738 | INFO     | diamonds.model:preprocess_data:87 - Data preprocessed using the preprocessing pipeline
2026-03-10 15:56:55.738 | INFO     | diamonds.model:preprocess_data:88 - Shape of the preprocessed data: (43136, 23) compared to the original data: (43136, 9)
2026-03-10 15:56:55.749 | INFO     | diamonds.model:preprocess_data:87 - Data preprocessed using the preprocessing pipeline
2026-03-10 15:56:55.750 | INFO     | diamonds.model:preprocess_data:88 - Shape of the preprocessed data: (10784, 23) compared to the original data: (10784, 9)
2026-03-10 15:56:55.751 | INFO     | diamonds.model:create_model:40 - Creating a random forest regression model
2026-03-10 15:57:06.846 | INFO     | diamonds.model:train_model:95 - Time to train the model: 11.093 sec
2026-03-10 15:57:07.012 | INFO     | diamonds.model:predict:128 - Predictions made

{'mape': 0.07135465990846582,
 'mae': 288.5318700752241,
 'mse': 352990.34937371855,
 'r2': 0.977982071407797}

In [20]:
dict_metrics = evaluate_model(model_, X_test_scaled, y_test)
for metric_name, metric_value in dict_metrics.items():
    mlflow.log_metric(metric_name, metric_value)

2026-03-10 12:26:39.012 | INFO     | diamonds.model:evaluate_model:104 - Model evaluation completed
2026-03-10 12:26:39.013 | INFO     | diamonds.model:evaluate_model:105 - Model scores: {'mape': 0.07388831312960824, 'mae': 298.2490601608621, 'mse': 385190.48617278325, 'r2': 0.9759735736855255}


In [14]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

mlflow.sklearn.autolog(max_tuning_runs=10)


params = {
    "n_estimators": [100],
    "max_depth": [20, 25, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [2, 5, 10],
}
mlflow.end_run()
with mlflow.start_run(run_name="RF Hyperparameter Tuning"):
    model_ = RandomForestRegressor()
    grid_search = GridSearchCV(estimator=model_, param_grid=params, cv=5, n_jobs=-1, verbose=4)
    grid_search.fit(X_train_scaled, y_train)
    
    dict_metrics = evaluate_model(grid_search.best_estimator_, X_test_scaled, y_test)
    for metric_name, metric_value in dict_metrics.items():
        mlflow.log_metric(metric_name, metric_value)
    mlflow.log_param("best_params", grid_search.best_params_)


🏃 View run angry-calf-761 at: http://localhost:5000/#/experiments/1/runs/60c26c1b049e47c89f408c49f8ef54e6
🧪 View experiment at: http://localhost:5000/#/experiments/1
Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV 5/5] END max_depth=20, min_samples_leaf=2, min_samples_split=5, n_estimators=100;, score=0.974 total time=  10.6s
[CV 1/5] END max_depth=20, min_samples_leaf=2, min_samples_split=5, n_estimators=100;, score=0.973 total time=  10.7s
[CV 4/5] END max_depth=20, min_samples_leaf=2, min_samples_split=5, n_estimators=100;, score=0.973 total time=  10.7s
[CV 1/5] END max_depth=20, min_samples_leaf=2, min_samples_split=2, n_estimators=100;, score=0.973 total time=  10.9s
[CV 3/5] END max_depth=20, min_samples_leaf=2, min_samples_split=2, n_estimators=100;, score=0.972 total time=  11.0s
[CV 2/5] END max_depth=20, min_samples_leaf=2, min_samples_split=10, n_estimators=100;, score=0.974 total time=  10.9s
[CV 4/5] END max_depth=20, min_samples_leaf=2, min_samples_spli

2026/03/10 14:40:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/10 14:40:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/10 14:41:03 INFO mlflow.sklearn.utils: Logging the 10 best runs, 17 runs will be omitted.


🏃 View run skillful-snipe-710 at: http://localhost:5000/#/experiments/1/runs/eba0886d47dc4a66807b89ea9150883c
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run lyrical-fly-11 at: http://localhost:5000/#/experiments/1/runs/c6b6ab1082254c84b87439274b6dcfa1
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run painted-midge-455 at: http://localhost:5000/#/experiments/1/runs/452b9306a8da4ab09541c61331c6eea3
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run bemused-fly-2 at: http://localhost:5000/#/experiments/1/runs/3816edb81450467ea7a9f5c69a88ebed
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run adaptable-fish-10 at: http://localhost:5000/#/experiments/1/runs/10ce09ad9e5144c79b4ecc20ea068d94
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run indecisive-eel-169 at: http://localhost:5000/#/experiments/1/runs/dafc5e6d48214d6eb80781c999605d76
🧪 View experiment at: http://localhost:5000/#/exper

2026-03-10 14:41:04.953 | INFO     | diamonds.model:evaluate_model:104 - Model evaluation completed
2026-03-10 14:41:04.954 | INFO     | diamonds.model:evaluate_model:105 - Model scores: {'mape': 0.0715306975901693, 'mae': 289.85568212421225, 'mse': 356972.1098107043, 'r2': 0.977733707345923}


🏃 View run RF Hyperparameter Tuning at: http://localhost:5000/#/experiments/1/runs/11ab3f7bb72048d3a8b118a7542d91b1
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [10]:
client = mlflow.MlflowClient()

model_name = "diamonds_model"
alias = "prod"


champion_version = mlflow.sklearn.load_model(f"models:/{model_name}@{alias}")

evaluate_model(champion_version, X_test_scaled, y_test)

2026-03-10 15:19:03.491 | INFO     | diamonds.model:evaluate_model:105 - Model evaluation completed
2026-03-10 15:19:03.492 | INFO     | diamonds.model:evaluate_model:106 - Model scores: {'mape': 0.07046109578637513, 'mae': 284.39159290131266, 'mse': 340921.04264397745, 'r2': 0.9787348997335694}


{'mape': 0.07046109578637513,
 'mae': 284.39159290131266,
 'mse': 340921.04264397745,
 'r2': 0.9787348997335694}

In [14]:
y_pred, scores = run_model_mkflow(X_test_scaled, y_test)

2026-03-10 15:28:11.238 | INFO     | diamonds.model:run_model_mkflow:4 - Model properly loaded from mlflow model registry
2026-03-10 15:28:11.624 | INFO     | diamonds.model:predict:127 - Predictions made using the trained model
2026-03-10 15:28:11.627 | INFO     | diamonds.model:evaluate_model:7 - Model evaluation completed
2026-03-10 15:28:11.628 | INFO     | diamonds.model:evaluate_model:8 - Model scores: {'mape': 0.07046109578637513, 'mae': 284.39159290131266, 'mse': 340921.04264397745, 'r2': 0.9787348997335694}
2026-03-10 15:28:11.920 | INFO     | diamonds.model:run_model_mkflow:9 - Model evaluation scores logged to mlflow


In [ ]:
list_index = random.sample(range(len(df_clean)), 10)
for idx in list_index:
    X_df = df_clean.drop(columns=["price"], axis=1).copy()
    test_X = pd.DataFrame([X_df.iloc[idx]])
    print(test_X, test_X.shape)
    test_y = df_clean.iloc[idx]["price"]
    pred = predict_values(test_X)
    
    print(f"True value: {test_y}, Predicted value: {pred}, difference = {abs(test_y - pred)[0]:.2f}, relative error = {abs(test_y - pred)[0]/test_y:.2%}") 

       carat      cut color clarity  depth  table     x     y     z
47268    0.7  Premium     J    VVS1   60.9   58.0  5.74  5.79  3.51 (1, 9)


2026-03-10 16:08:43.052 | INFO     | diamonds.predict:predict_values:13 - Model loaded successfully for prediction
2026-03-10 16:08:43.058 | INFO     | diamonds.model:preprocess_data:87 - Data preprocessed using the preprocessing pipeline
2026-03-10 16:08:43.059 | INFO     | diamonds.model:preprocess_data:88 - Shape of the preprocessed data: (1, 23) compared to the original data: (1, 9)
2026-03-10 16:08:43.060 | INFO     | diamonds.predict:predict_values:15 - Data preprocessed successfully for prediction
2026-03-10 16:08:43.073 | INFO     | diamonds.model:predict:128 - Predictions made using the trained model
2026-03-10 16:08:43.074 | INFO     | diamonds.predict:predict_values:18 - Predictions made for prediction function


True value: 1844, Predicted value: [1878.1875], difference = [34.1875]
       carat      cut color clarity  depth  table     x     y     z
26007    2.2  Premium     I     SI2   63.0   59.0  8.22  8.19  5.17 (1, 9)


2026-03-10 16:08:44.083 | INFO     | diamonds.predict:predict_values:13 - Model loaded successfully for prediction
2026-03-10 16:08:44.089 | INFO     | diamonds.model:preprocess_data:87 - Data preprocessed using the preprocessing pipeline
2026-03-10 16:08:44.090 | INFO     | diamonds.model:preprocess_data:88 - Shape of the preprocessed data: (1, 23) compared to the original data: (1, 9)
2026-03-10 16:08:44.091 | INFO     | diamonds.predict:predict_values:15 - Data preprocessed successfully for prediction
2026-03-10 16:08:44.106 | INFO     | diamonds.model:predict:128 - Predictions made using the trained model
2026-03-10 16:08:44.106 | INFO     | diamonds.predict:predict_values:18 - Predictions made for prediction function


True value: 15238, Predicted value: [14564.93612724], difference = [673.06387276]
       carat      cut color clarity  depth  table    x     y    z
16614   1.52  Premium     I     SI1   58.8   61.0  7.5  7.46  4.4 (1, 9)


...

True value: 14139, Predicted value: [13947.11869248], difference = [191.88130752]
       carat    cut color clarity  depth  table     x     y     z
26683   0.33  Ideal     E     SI2   62.2   54.0  4.44  4.46  2.77 (1, 9)


2026-03-10 16:08:51.489 | INFO     | diamonds.predict:predict_values:13 - Model loaded successfully for prediction
2026-03-10 16:08:51.496 | INFO     | diamonds.model:preprocess_data:87 - Data preprocessed using the preprocessing pipeline
2026-03-10 16:08:51.497 | INFO     | diamonds.model:preprocess_data:88 - Shape of the preprocessed data: (1, 23) compared to the original data: (1, 9)
2026-03-10 16:08:51.497 | INFO     | diamonds.predict:predict_values:15 - Data preprocessed successfully for prediction
2026-03-10 16:08:51.510 | INFO     | diamonds.model:predict:128 - Predictions made using the trained model
2026-03-10 16:08:51.510 | INFO     | diamonds.predict:predict_values:18 - Predictions made for prediction function


True value: 427, Predicted value: [439.53972223], difference = [12.53972223]
      carat        cut color clarity  depth  table     x     y     z
9814   0.91  Very Good     H     VS1   59.4   63.0  6.32  6.28  3.74 (1, 9)


2026-03-10 16:08:52.521 | INFO     | diamonds.predict:predict_values:13 - Model loaded successfully for prediction
2026-03-10 16:08:52.527 | INFO     | diamonds.model:preprocess_data:87 - Data preprocessed using the preprocessing pipeline
2026-03-10 16:08:52.527 | INFO     | diamonds.model:preprocess_data:88 - Shape of the preprocessed data: (1, 23) compared to the original data: (1, 9)
2026-03-10 16:08:52.528 | INFO     | diamonds.predict:predict_values:15 - Data preprocessed successfully for prediction
2026-03-10 16:08:52.538 | INFO     | diamonds.model:predict:128 - Predictions made using the trained model
2026-03-10 16:08:52.539 | INFO     | diamonds.predict:predict_values:18 - Predictions made for prediction function


True value: 4670, Predicted value: [4429.00960347], difference = [240.99039653]
